# Pair 2 & Pair 4: ResNet18 on MNIST and CIFAR-10 (Topic 30)
Same model (ResNet18) on both datasets, so the code is shared and only the
dataset-loading cells differ between Part A (MNIST) and Part B (CIFAR-10).

Run on Kaggle: **Settings -> Accelerator -> GPU T4 x2**, **Settings -> Internet -> On**

In [ ]:
import os
import csv
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.utils import save_image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

IS_KAGGLE = os.path.exists("/kaggle")
BATCH_SIZE = 128
LR = 1e-3
PGD_STEPS = 20

## Shared code (used by both MNIST and CIFAR-10 sections)

`build_resnet18()` reuses torchvision's ResNet18 but swaps the stem (`conv1` +
`maxpool`) for a version that does not aggressively downsample -- the
standard fix for using ResNet18 on small (28x28 / 32x32) images instead of
its native 224x224 ImageNet input. Trained from scratch (no pretrained
weights), and we skip mean/std normalization so pixels stay in [0, 1] and
epsilon keeps the same simple meaning as in the earlier MNIST/CIFAR-10 + CNN
experiments.

The FGSM/PGD attack functions themselves are identical to the ones used for
the plain CNNs -- the attack code does not care what model it's attacking.

In [ ]:
def build_resnet18(num_classes=10, in_channels=3):
    model = models.resnet18(weights=None, num_classes=num_classes)
    model.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model.to(DEVICE)


def count_params(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
def train(model, loader, optimizer, epochs):
    model.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
        print(f"Epoch {epoch}/{epochs} - train loss: {total_loss / len(loader.dataset):.4f}")

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
    return correct / len(loader.dataset)

In [ ]:
def fgsm_attack(model, images, labels, epsilon):
    images = images.clone().detach().to(DEVICE)
    labels = labels.to(DEVICE)
    images.requires_grad = True

    outputs = model(images)
    loss = F.cross_entropy(outputs, labels)

    model.zero_grad()
    loss.backward()

    grad_sign = images.grad.data.sign()
    adv_images = images + epsilon * grad_sign
    return torch.clamp(adv_images, 0, 1).detach()


def pgd_attack(model, images, labels, epsilon, alpha, num_steps=PGD_STEPS):
    images = images.clone().detach().to(DEVICE)
    labels = labels.to(DEVICE)
    adv_images = images.clone().detach()

    for _ in range(num_steps):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = F.cross_entropy(outputs, labels)

        model.zero_grad()
        loss.backward()

        grad_sign = adv_images.grad.data.sign()
        adv_images = adv_images.detach() + alpha * grad_sign

        perturbation = torch.clamp(adv_images - images, -epsilon, epsilon)
        adv_images = torch.clamp(images + perturbation, 0, 1).detach()

    return adv_images


@torch.no_grad()
def _accuracy_on(model, images, labels):
    preds = model(images).argmax(dim=1)
    return (preds == labels).sum().item()


def evaluate_fgsm(model, loader, epsilon):
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        if epsilon == 0.0:
            correct += _accuracy_on(model, images, labels)
        else:
            adv_images = fgsm_attack(model, images, labels, epsilon)
            correct += _accuracy_on(model, adv_images, labels)
        total += labels.size(0)
    return correct / total


def evaluate_pgd(model, loader, epsilon, alpha, fgsm_eps0_acc):
    if epsilon == 0.0:
        return fgsm_eps0_acc
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        adv_images = pgd_attack(model, images, labels, epsilon, alpha)
        correct += _accuracy_on(model, adv_images, labels)
        total += labels.size(0)
    return correct / total

In [ ]:
def run_epsilon_sweep(model, loader, epsilons, alpha):
    results = []
    fgsm_results = {}
    print("epsilon | FGSM accuracy | PGD accuracy")
    print("--------|----------------|-------------")
    for eps in epsilons:
        fgsm_acc = evaluate_fgsm(model, loader, eps)
        fgsm_results[eps] = fgsm_acc
        pgd_acc = evaluate_pgd(model, loader, eps, alpha, fgsm_results[0.0])
        print(f"{eps:>7.2f} | {fgsm_acc * 100:13.2f}% | {pgd_acc * 100:11.2f}%")
        results.append({
            "epsilon": eps,
            "fgsm_accuracy": round(fgsm_acc * 100, 2),
            "pgd_accuracy": round(pgd_acc * 100, 2),
        })
    return results


def save_sample_images(model, loader, attack_fn, filename_prefix, epsilon, class_names=None):
    images, labels = next(iter(loader))
    images, labels = images[:6].to(DEVICE), labels[:6].to(DEVICE)
    adv_images = attack_fn(model, images, labels, epsilon)

    with torch.no_grad():
        clean_preds = model(images).argmax(dim=1)
        adv_preds = model(adv_images).argmax(dim=1)

    def name(i):
        return class_names[i] if class_names else str(i)

    print("\nSample predictions (true -> clean_pred -> adv_pred):")
    for i in range(6):
        true_c = name(labels[i].item())
        clean_c = name(clean_preds[i].item())
        adv_c = name(adv_preds[i].item())
        tag = "FOOLED" if adv_preds[i] != labels[i] else "still correct"
        print(f"  true={true_c:6s} clean_pred={clean_c:6s} adv_pred={adv_c:6s}  {tag}")

    save_image(images, f"{filename_prefix}_clean.png", nrow=6)
    save_image(adv_images, f"{filename_prefix}_adversarial.png", nrow=6)
    print(f"Saved {filename_prefix}_clean.png and {filename_prefix}_adversarial.png")


def save_results_csv(results, csv_path):
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["epsilon", "fgsm_accuracy", "pgd_accuracy"])
        writer.writeheader()
        writer.writerows(results)
    print(f"Saved {csv_path}")


def download_file(path):
    if not IS_KAGGLE:
        from google.colab import files
        files.download(path)
    else:
        print(f"On Kaggle: find {path} in the Output panel (right side) and download it from there.")

## Part A: MNIST + ResNet18 (Pair 2)

In [ ]:
MNIST_EPOCHS = 8
MNIST_EPSILONS = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
MNIST_PGD_ALPHA = 0.01
MNIST_MODEL_PATH = "resnet18_mnist.pt"

transform = transforms.ToTensor()
mnist_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
mnist_train_loader = DataLoader(mnist_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
mnist_test_loader = DataLoader(mnist_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
mnist_model = build_resnet18(num_classes=10, in_channels=1)
print(f"ResNet18 (MNIST) parameters: {count_params(mnist_model):,}")

optimizer = torch.optim.Adam(mnist_model.parameters(), lr=LR)
train(mnist_model, mnist_train_loader, optimizer, MNIST_EPOCHS)

mnist_clean_acc = evaluate(mnist_model, mnist_test_loader)
print(f"\nClean test accuracy: {mnist_clean_acc * 100:.2f}%")

torch.save(mnist_model.state_dict(), MNIST_MODEL_PATH)
print(f"Model saved to {MNIST_MODEL_PATH}")

In [ ]:
mnist_results = run_epsilon_sweep(mnist_model, mnist_test_loader, MNIST_EPSILONS, MNIST_PGD_ALPHA)

In [ ]:
save_sample_images(mnist_model, mnist_test_loader, fgsm_attack, "resnet18_mnist_fgsm", epsilon=0.2)
save_sample_images(
    mnist_model, mnist_test_loader,
    lambda m, i, l, e: pgd_attack(m, i, l, e, MNIST_PGD_ALPHA),
    "resnet18_mnist_pgd", epsilon=0.2,
)

In [ ]:
save_results_csv(mnist_results, "resnet18_mnist_fgsm_pgd_results.csv")
download_file("resnet18_mnist_fgsm_pgd_results.csv")
download_file(MNIST_MODEL_PATH)

## Part B: CIFAR-10 + ResNet18 (Pair 4)

Note: CIFAR-10's download server (cs.toronto.edu) can be slow/congested on
Kaggle -- if the download looks stuck at a very low speed, interrupt the
cell and re-run it; it is usually much faster on retry. For an unattended
run, use **Save Version -> Save & Run All (Commit)** rather than staying in
the interactive session.

In [ ]:
CIFAR_EPOCHS = 8
CIFAR_EPSILONS = [0.0, 0.01, 0.02, 0.05, 0.1, 0.15, 0.2]
CIFAR_PGD_ALPHA = 0.005
CIFAR_MODEL_PATH = "resnet18_cifar10.pt"
CIFAR_CLASSES = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

cifar_train = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
cifar_test = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)
cifar_train_loader = DataLoader(cifar_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
cifar_test_loader = DataLoader(cifar_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
cifar_model = build_resnet18(num_classes=10, in_channels=3)
print(f"ResNet18 (CIFAR-10) parameters: {count_params(cifar_model):,}")

optimizer = torch.optim.Adam(cifar_model.parameters(), lr=LR)
train(cifar_model, cifar_train_loader, optimizer, CIFAR_EPOCHS)

cifar_clean_acc = evaluate(cifar_model, cifar_test_loader)
print(f"\nClean test accuracy: {cifar_clean_acc * 100:.2f}%")

torch.save(cifar_model.state_dict(), CIFAR_MODEL_PATH)
print(f"Model saved to {CIFAR_MODEL_PATH}")

In [ ]:
cifar_results = run_epsilon_sweep(cifar_model, cifar_test_loader, CIFAR_EPSILONS, CIFAR_PGD_ALPHA)

In [ ]:
save_sample_images(cifar_model, cifar_test_loader, fgsm_attack, "resnet18_cifar10_fgsm", epsilon=0.05, class_names=CIFAR_CLASSES)
save_sample_images(
    cifar_model, cifar_test_loader,
    lambda m, i, l, e: pgd_attack(m, i, l, e, CIFAR_PGD_ALPHA),
    "resnet18_cifar10_pgd", epsilon=0.05, class_names=CIFAR_CLASSES,
)

In [ ]:
save_results_csv(cifar_results, "resnet18_cifar10_fgsm_pgd_results.csv")
download_file("resnet18_cifar10_fgsm_pgd_results.csv")
download_file(CIFAR_MODEL_PATH)

## Summary

Prints parameter counts and clean accuracy for both models side by side --
useful for the "model capacity" discussion in the report (ResNet18 vs the
earlier Simple CNN).

In [ ]:
print(f"MNIST  + ResNet18: {count_params(mnist_model):,} params, clean acc {mnist_clean_acc*100:.2f}%")
print(f"CIFAR10 + ResNet18: {count_params(cifar_model):,} params, clean acc {cifar_clean_acc*100:.2f}%")